# VLM RAG Pipeline — Vision-Aware Document Q&A

This notebook demonstrates a **Vision Language Model (VLM) RAG** pipeline that
goes beyond simple text extraction. Instead of just pulling text from PDFs, we:

1. **Render** each page as an image
2. **Describe** the page with a VLM (captures tables, charts, figures)
3. **Embed** both the extracted text AND the VLM description (dual embeddings)
4. **Search** across both embedding types for better retrieval
5. **Answer** using the VLM with the actual page images — not lossy text

**Why VLM RAG?** Standard text extraction misses figures, charts, tables, and
layout. A VLM *sees* the page the way a human would.

Everything runs **locally** using [Ollama](https://ollama.com/).

In [ ]:
%pip install pymupdf sqlite-vec requests rich gradio --quiet

## Step 0: Configuration

VLM RAG uses **three separate models**:

| Role | Local (Ollama) | Azure |
|------|---------------|-------|
| Embeddings | `phi3:mini` | `text-embedding-ada-002` |
| Chat / Reranking | `phi3:mini` | `gpt-4o` |
| Vision (VLM) | `qwen3.5:4b` | `gpt-4o` |

Make sure you have the Ollama models pulled:
```bash
ollama pull phi3:mini
ollama pull qwen3.5:4b
```

In [ ]:
import os
import struct
import base64
import sqlite3
from pathlib import Path

import requests

# Load .env from parent vlm-rag/ directory if it exists
_env_path = Path("../vlm-rag/.env")
if _env_path.exists():
    for line in _env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

# --- Provider ---
PROVIDER = os.environ.get("PROVIDER", "local")

# --- Ollama ---
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_EMBED_MODEL = os.environ.get("OLLAMA_EMBED_MODEL", "phi3:mini")
OLLAMA_CHAT_MODEL = os.environ.get("OLLAMA_CHAT_MODEL", "phi3:mini")
OLLAMA_VLM_MODEL = os.environ.get("OLLAMA_VLM_MODEL", "qwen3.5:4b")

# --- Azure ---
AZURE_API_KEY = os.environ.get("AZURE_API_KEY", "")
AZURE_CHAT_BASE_URL = os.environ.get("AZURE_CHAT_BASE_URL", "").rstrip("/")
AZURE_CHAT_DEPLOYMENT = os.environ.get("AZURE_CHAT_DEPLOYMENT", "gpt-4o")
AZURE_VLM_BASE_URL = os.environ.get("AZURE_VLM_BASE_URL", "").rstrip("/")
AZURE_VLM_DEPLOYMENT = os.environ.get("AZURE_VLM_DEPLOYMENT", "gpt-4o")
AZURE_EMBED_BASE_URL = os.environ.get("AZURE_EMBED_BASE_URL", "").rstrip("/")
AZURE_EMBED_DEPLOYMENT = os.environ.get("AZURE_EMBED_DEPLOYMENT", "text-embedding-ada-002")


def _azure_headers() -> dict:
    return {"Authorization": f"Bearer {AZURE_API_KEY}", "Content-Type": "application/json"}


def get_embedding(text: str) -> list[float]:
    if PROVIDER == "azure":
        r = requests.post(f"{AZURE_EMBED_BASE_URL}/embeddings", headers=_azure_headers(),
                          json={"model": AZURE_EMBED_DEPLOYMENT, "input": text})
        r.raise_for_status()
        return r.json()["data"][0]["embedding"]
    else:
        r = requests.post(f"{OLLAMA_BASE_URL}/api/embed",
                          json={"model": OLLAMA_EMBED_MODEL, "input": text})
        r.raise_for_status()
        return r.json()["embeddings"][0]


def chat(messages: list[dict], temperature: float = 0, max_tokens: int = 1024) -> str:
    """Text-only chat (used for reranking)."""
    if PROVIDER == "azure":
        r = requests.post(f"{AZURE_CHAT_BASE_URL}/chat/completions", headers=_azure_headers(),
                          json={"model": AZURE_CHAT_DEPLOYMENT, "messages": messages,
                                "temperature": temperature, "max_completion_tokens": max_tokens})
    else:
        r = requests.post(f"{OLLAMA_BASE_URL}/v1/chat/completions",
                          json={"model": OLLAMA_CHAT_MODEL, "messages": messages,
                                "temperature": temperature, "max_completion_tokens": max_tokens})
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def vlm_chat(messages: list[dict], temperature: float = 0, max_tokens: int = 1024) -> str:
    """Vision-capable chat (supports image content parts)."""
    if PROVIDER == "azure":
        r = requests.post(f"{AZURE_VLM_BASE_URL}/chat/completions", headers=_azure_headers(),
                          json={"model": AZURE_VLM_DEPLOYMENT, "messages": messages,
                                "temperature": temperature, "max_completion_tokens": max_tokens})
    else:
        r = requests.post(f"{OLLAMA_BASE_URL}/v1/chat/completions",
                          json={"model": OLLAMA_VLM_MODEL, "messages": messages,
                                "temperature": temperature, "max_completion_tokens": max_tokens})
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def serialize_float32(vector: list[float]) -> bytes:
    return struct.pack(f"{len(vector)}f", *vector)


print(f"Provider : {PROVIDER}")
if PROVIDER == "azure":
    print(f"Embed    : Azure ({AZURE_EMBED_DEPLOYMENT})")
    print(f"Chat     : Azure ({AZURE_CHAT_DEPLOYMENT})")
    print(f"VLM      : Azure ({AZURE_VLM_DEPLOYMENT})")
else:
    print(f"Embed    : Ollama ({OLLAMA_EMBED_MODEL})")
    print(f"Chat     : Ollama ({OLLAMA_CHAT_MODEL})")
    print(f"VLM      : Ollama ({OLLAMA_VLM_MODEL})")

## Step 1: Ingest — PDF to Page Images + VLM Descriptions

For each PDF page we:
1. Extract the raw text with PyMuPDF
2. Render the page as a PNG image (150 DPI)
3. Ask the VLM to **describe** the page — capturing figures, charts, tables, and equations

All stored in a single SQLite database (one row per page, no chunking).

> **Note**: This step calls the VLM once per page, so it may take several minutes
> depending on the number of pages and your hardware.

In [ ]:
import fitz  # pymupdf

DOCS_DIR = Path("../vlm-rag/docs")
DB_FILE = Path("vlm_rag_notebook.db")
DPI = 150


def render_page_to_base64(page: fitz.Page) -> str:
    """Render a PDF page to a base64-encoded PNG string."""
    pix = page.get_pixmap(dpi=DPI)
    return base64.b64encode(pix.tobytes("png")).decode("ascii")


def describe_image(image_b64: str, page_num: int) -> str:
    """Ask the VLM to describe a page image."""
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"This is page {page_num} of a document. Describe this page concisely:\n"
                        "1. If there are diagrams or figures: state the exact title/label and briefly describe what the diagram shows.\n"
                        "2. If there are charts or graphs: state the title, axis labels, and what trend or comparison is shown.\n"
                        "3. If there are tables: state the title and what the columns/rows represent.\n"
                        "4. If there are equations: write them out.\n"
                        "5. Summarize the main text content in 2-3 sentences.\n"
                        "Focus on extracting structured information, not narrating layout."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{image_b64}"},
                },
            ],
        },
    ]
    return vlm_chat(messages, max_tokens=512)


# --- Find PDFs ---
pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {DOCS_DIR.resolve()}.\n"
        "Place a PDF there first."
    )
print(f"Found {len(pdf_files)} PDF(s): {[p.name for p in pdf_files]}")

# --- Fresh DB ---
if DB_FILE.exists():
    DB_FILE.unlink()

db = sqlite3.connect(str(DB_FILE))
db.execute("""
    CREATE TABLE pages (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        text TEXT NOT NULL,
        description TEXT NOT NULL,
        image_b64 TEXT NOT NULL,
        source TEXT NOT NULL,
        page INTEGER NOT NULL
    )
""")

all_pages = []
for pdf_path in pdf_files:
    print(f"\nProcessing: {pdf_path.name}")
    doc = fitz.open(pdf_path)
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        if not text.strip():
            continue
        image_b64 = render_page_to_base64(page)
        all_pages.append((pdf_path.name, page_num + 1, text, image_b64))
    doc.close()
    print(f"  Extracted {len(all_pages)} pages so far")

print(f"\nDescribing {len(all_pages)} pages with VLM...")
for i, (source, page_num, text, image_b64) in enumerate(all_pages, 1):
    print(f"  [{i}/{len(all_pages)}] {source} page {page_num}...", end=" ")
    description = describe_image(image_b64, page_num)
    db.execute(
        "INSERT INTO pages (text, description, image_b64, source, page) VALUES (?, ?, ?, ?, ?)",
        (text, description, image_b64, source, page_num),
    )
    print("done")

db.commit()
print(f"\nTotal pages stored: {len(all_pages)}")

# Display summary
rows = db.execute("SELECT id, page, LENGTH(image_b64), description FROM pages").fetchall()
for row in rows:
    print(f"  Page {row[1]} | Image: {row[2] // 1024} KB | Desc: {row[3][:100].replace(chr(10), ' ')}...")

## Step 2: Embed — Dual Embeddings (Text + Description)

We create **two embeddings per page**:
1. One from the **extracted text**
2. One from the **VLM description** (chart titles, axis labels, figure summaries)

Both point back to the same page. At query time we search both, so a question
about "vacancies-to-unemployment ratio" can match the description embedding
(which has the chart title) even if the raw text doesn't mention it.

Pages with no visual content (no diagrams/charts) skip the description embedding
to avoid noise.

In [ ]:
import sqlite_vec

# Reopen DB with sqlite-vec
db.close()
db = sqlite3.connect(str(DB_FILE))
db.enable_load_extension(True)
sqlite_vec.load(db)
db.enable_load_extension(False)

pages = db.execute("SELECT id, text, description FROM pages ORDER BY id").fetchall()
print(f"Embedding {len(pages)} pages (dual embeddings: text + description)...")

# Discover dimension
first_vec = get_embedding(pages[0][1])
dim = len(first_vec)
print(f"Embedding dimension: {dim}")

# Create index tables
db.execute("DROP TABLE IF EXISTS embed_index")
db.execute("""
    CREATE TABLE embed_index (
        embed_id INTEGER PRIMARY KEY,
        page_id INTEGER NOT NULL,
        source_type TEXT NOT NULL
    )
""")

db.execute("DROP TABLE IF EXISTS vec_pages")
db.execute(f"""
    CREATE VIRTUAL TABLE vec_pages USING vec0(
        embed_id INTEGER PRIMARY KEY,
        embedding FLOAT[{dim}]
    )
""")

embed_id = 0
skipped = 0
for page_id, text, description in pages:
    # Embedding 1: extracted text
    text_vec = get_embedding(text)
    db.execute("INSERT INTO embed_index (embed_id, page_id, source_type) VALUES (?, ?, ?)",
               (embed_id, page_id, "text"))
    db.execute("INSERT INTO vec_pages (embed_id, embedding) VALUES (?, ?)",
               (embed_id, serialize_float32(text_vec)))
    embed_id += 1

    # Embedding 2: VLM description (only if page has visual content)
    desc_lower = description.lower()
    has_visuals = not ("no diagrams" in desc_lower or "no charts" in desc_lower or "there are no" in desc_lower)
    if has_visuals:
        desc_vec = get_embedding(description)
        db.execute("INSERT INTO embed_index (embed_id, page_id, source_type) VALUES (?, ?, ?)",
                   (embed_id, page_id, "description"))
        db.execute("INSERT INTO vec_pages (embed_id, embedding) VALUES (?, ?)",
                   (embed_id, serialize_float32(desc_vec)))
        embed_id += 1
    else:
        skipped += 1

    print(f"  Page {page_id} embedded", end="\r")

db.commit()
print(f"\nDone! {embed_id} total embeddings ({len(pages)} text + {embed_id - len(pages)} description)")
print(f"Skipped {skipped} text-only pages (no visual content)")

# Stats
print(f"\nDatabase stats:")
print(f"  pages       : {len(pages)} rows")
print(f"  embed_index : {embed_id} rows")
print(f"  vec_pages   : {embed_id} rows ({dim}-dimensional vectors)")

## Step 3: Query — VLM-Powered Answer

The key difference from text RAG: we send the actual **page images** to a Vision LLM
for the final answer. The VLM sees tables, charts, and layout — not lossy extracted text.

Pipeline:
1. Embed the question (text embedding)
2. Vector search across both text and description embeddings
3. Deduplicate by page (keep best match per page)
4. Send page images + descriptions to VLM
5. Display the answer

In [ ]:
TOP_K = 3


def search_similar(db, query_vec, top_k=3):
    """Search across text and description embeddings, deduplicate by page."""
    rows = db.execute(
        """
        SELECT v.embed_id, v.distance, e.page_id, e.source_type,
               p.text, p.description, p.image_b64, p.source, p.page
        FROM vec_pages v
        JOIN embed_index e ON e.embed_id = v.embed_id
        JOIN pages p ON p.id = e.page_id
        WHERE v.embedding MATCH ?
            AND v.k = ?
        ORDER BY v.distance
        """,
        (serialize_float32(query_vec), top_k * 2),
    ).fetchall()

    seen_pages = set()
    results = []
    for r in rows:
        page_id = r[2]
        if page_id in seen_pages:
            continue
        seen_pages.add(page_id)
        results.append({
            "id": page_id, "distance": r[1], "matched_on": r[3],
            "text": r[4], "description": r[5],
            "image_b64": r[6], "source": r[7], "page": r[8],
        })
        if len(results) >= top_k:
            break
    return results


def build_vlm_messages(question, results):
    """Build messages with page images for the VLM."""
    content_parts = []
    for r in results:
        content_parts.append({
            "type": "text",
            "text": f"[Page {r['page']} from {r['source']}]\nDescription: {r['description']}",
        })
        content_parts.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{r['image_b64']}"},
        })
    content_parts.append({"type": "text", "text": f"\n---\n\nQuestion: {question}"})

    return [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the user's question based on "
                "the provided page images ONLY, not internal knowledge. If the images "
                "don't contain enough information, say so. "
                "Cite which page the information comes from when possible."
            ),
        },
        {"role": "user", "content": content_parts},
    ]


print("Query functions defined.")

In [ ]:
# --- Run a sample query ---
question = "Summarize the key chart on unemployment ratio"

print(f"Question: {question}\n")

# 1. Embed the question
query_vec = get_embedding(question)
print(f"Query vector dimension: {len(query_vec)}")

# 2. Search
results = search_similar(db, query_vec, top_k=TOP_K)

print(f"\nRetrieved {len(results)} pages:")
for i, r in enumerate(results, 1):
    print(f"  [{i}] Page {r['page']} (dist={r['distance']:.4f}, matched on: {r['matched_on']})")
    print(f"      {r['description'][:120].replace(chr(10), ' ')}...")

# 3. Send to VLM
print(f"\nSending to VLM...")
messages = build_vlm_messages(question, results)
answer = vlm_chat(messages)

print(f"\n{'=' * 80}")
print("ANSWER")
print(f"{'=' * 80}")
print(answer)

## Step 4: Query with Reranking

Vector search is fast but imprecise — it matches by embedding distance, which
can miss nuance. A **reranker** reads each candidate carefully and scores how
relevant it actually is to your question.

Pipeline:
1. Vector search with a wider net (top 6 candidates)
2. LLM scores each candidate's relevance 0-10
3. Keep top 3 after reranking
4. Send those pages to the VLM for the final answer

In [ ]:
INITIAL_K = 6   # wider net for initial retrieval
FINAL_K = 3     # keep top 3 after reranking


def rerank(question: str, candidates: list[dict]) -> list[dict]:
    """Rerank candidates using LLM as a cross-encoder scorer."""
    scored = []
    for c in candidates:
        context = c["description"]
        score_text = chat(
            [{"role": "user", "content": (
                "Rate relevance of the document to the question. "
                "Reply with ONLY a single integer 0-10.\n\n"
                f"Question: {question}\n\n"
                f"Document: {context}\n\n"
                "Relevance score (0-10):"
            )}],
            max_tokens=4,
        ).strip()

        score = 0.0
        for token in score_text.replace("/", " ").split():
            try:
                val = float(token)
                if 0 <= val <= 10:
                    score = val
                    break
            except ValueError:
                continue

        scored.append({**c, "rerank_score": score})

    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored


# --- Run reranked query ---
question_rr = "Summarize the key chart on unemployment ratio"

print(f"Question: {question_rr}\n")

# 1. Embed
query_vec = get_embedding(question_rr)

# 2. Vector search — wider net
candidates = search_similar(db, query_vec, top_k=INITIAL_K)
print(f"Initial candidates (vector search, top {INITIAL_K}):")
for i, r in enumerate(candidates, 1):
    print(f"  [{i}] Page {r['page']} (dist={r['distance']:.4f}) {r['description'][:80].replace(chr(10), ' ')}...")

# 3. Rerank
print(f"\nReranking with LLM...")
reranked = rerank(question_rr, candidates)

old_ranks = {r["page"]: i + 1 for i, r in enumerate(candidates)}
print(f"\nAfter reranking:")
for i, r in enumerate(reranked):
    marker = "*" if i < FINAL_K else " "
    print(f"  {marker} [{i+1}] Page {r['page']} (score={r['rerank_score']:.0f}, was #{old_ranks[r['page']]}) {r['description'][:60].replace(chr(10), ' ')}...")
print(f"  (* = selected for VLM, top {FINAL_K})")

# 4. Send top results to VLM
final_results = reranked[:FINAL_K]
print(f"\nSending to VLM...")
messages = build_vlm_messages(question_rr, final_results)
answer = vlm_chat(messages)

print(f"\n{'=' * 80}")
print("ANSWER (with reranking)")
print(f"{'=' * 80}")
print(answer)

## Try your own question

Change the `question` variable below and re-run the cell.

In [ ]:
# Change this to any question about your ingested documents
question = "What are the main findings of the document?"

# --- Retrieve & Generate (with reranking) ---
query_vec = get_embedding(question)
candidates = search_similar(db, query_vec, top_k=INITIAL_K)
reranked = rerank(question, candidates)
final_results = reranked[:FINAL_K]

print(f"Question: {question}\n")
print(f"Top {FINAL_K} pages after reranking:")
for i, r in enumerate(final_results, 1):
    print(f"  {i}. Page {r['page']} (score={r['rerank_score']:.0f}) {r['description'][:80].replace(chr(10), ' ')}...")

messages = build_vlm_messages(question, final_results)
answer = vlm_chat(messages)

print(f"\n{'=' * 80}")
print("ANSWER")
print(f"{'=' * 80}")
print(answer)

## Interactive Q&A with Gradio

Run the cell below to launch a **web interface** for querying your VLM RAG pipeline
interactively. Uses reranking for better results.

In [ ]:
import gradio as gr


def gradio_vlm_query(question: str) -> str:
    """VLM RAG query with reranking for Gradio interface."""
    if not question.strip():
        return "Please enter a question."
    try:
        query_vec = get_embedding(question)
        candidates = search_similar(db, query_vec, top_k=INITIAL_K)
        if not candidates:
            return "No matching pages found."

        reranked = rerank(question, candidates)
        final_results = reranked[:FINAL_K]

        messages = build_vlm_messages(question, final_results)
        answer = vlm_chat(messages)

        output = ["=== Retrieved Pages (after reranking) ==="]
        for i, r in enumerate(final_results, 1):
            desc_snippet = r["description"][:200].replace("\n", " ")
            output.append(
                f"\n[{i}] (score: {r['rerank_score']:.0f}) "
                f"{r['source']}, page {r['page']}\n    {desc_snippet}..."
            )
        output.append("\n\n=== Answer ===")
        output.append(answer)
        return "\n".join(output)
    except Exception as e:
        return f"Error: {e}"


demo = gr.Interface(
    fn=gradio_vlm_query,
    inputs=gr.Textbox(label="Your Question", placeholder="e.g., Summarize the key chart"),
    outputs=gr.Textbox(label="Answer", lines=15),
    title="VLM RAG Q&A",
    description="Ask questions about your ingested documents. Uses vision model to analyze page images.",
)
demo.launch()